# BioOptima — 3000-sample PyADM1ODE sweep (Colab, HRT-first)

Run this notebook on **Google Colab** to generate the primary training dataset (~**40–90 min** on 2 CPU workers for 3000 runs).

**Sampling (HRT-first):** Latin Hypercube on temperature (25–45 °C) and HRT (10–50 d); Dirichlet feed mixes; flows set so HRT is exact and **OLR** is derived (kept in 0.5–8.0 kg VS/m³/d to match the app sliders).

**Substrate YAMLs** define **diluted slurry** (10–20% TS) as fed to a CSTR — not raw feedstock. TS values calibrated against 21 Indian AD literature points so achieved OLR covers ~1.5–8.0 and CH₄ yields approach published observations.

**What you will see while it runs**
- A **tqdm** progress bar with ETA (sight)
- A **live status panel** (rows done, % converged so far, rolling CH₄, elapsed / ETA)
- **Two charts** that refresh: cumulative convergence rate and recent CH₄ outcomes
- **Short milestone tones** at 25 / 50 / 75 / 100% (hearing — may be muted until you interact with the page once in Colab)
- A **heartbeat** print every N simulations so the log still moves if plots lag

After completion, run the download cell and place the CSV at `data/generated/biogas_training_data.csv` in your local repo.

**Important:** The project must live **on Google Drive** (synced or zipped upload). You cannot use a Windows path like `D:\\major project` as `PROJECT_ROOT` in Colab.

In [ ]:
# 1. Mount Drive / set project root
#
# Colab runs in the cloud — it does NOT see D:\\ or C:\\ on your PC.
# Sync or zip-upload the whole repo to Google Drive, then point PROJECT_ROOT
# at that folder (forward slashes). If the path is wrong, this cell lists
# top-level folders under My Drive and tries to auto-find `src` + `pyADM1ODE`.
#
from google.colab import drive
drive.mount('/content/drive')

import os, sys

MY_DRIVE = '/content/drive/MyDrive'

# --- EDIT THIS if auto-detect fails (must be the folder that contains src/, pyADM1ODE/, data/) ---
PROJECT_ROOT = ''  # e.g. '/content/drive/MyDrive/major_project' (underscore) or '.../major project' (space)


def _is_project_root(path: str) -> bool:
    return os.path.isdir(path) and os.path.isdir(os.path.join(path, 'src')) and os.path.isdir(
        os.path.join(path, 'pyADM1ODE')
    )


def _find_project_under(start: str, max_depth: int = 4):
    """Breadth-first search for a folder containing src/ and pyADM1ODE/."""
    from collections import deque

    q = deque([(start, 0)])
    seen = set()
    while q:
        p, d = q.popleft()
        if p in seen or d > max_depth:
            continue
        seen.add(p)
        if _is_project_root(p):
            return p
        if not os.path.isdir(p):
            continue
        try:
            for name in os.listdir(p):
                if name.startswith('.'):
                    continue
                c = os.path.join(p, name)
                if os.path.isdir(c):
                    q.append((c, d + 1))
        except OSError:
            pass
    return None


if not PROJECT_ROOT or not os.path.isdir(PROJECT_ROOT):
    print('Top-level folders under My Drive (pick your project parent):')
    try:
        for name in sorted(os.listdir(MY_DRIVE)):
            p = os.path.join(MY_DRIVE, name)
            if os.path.isdir(p):
                print(' ', name)
    except OSError as e:
        print('Could not list My Drive:', e)

    found = _find_project_under(MY_DRIVE, max_depth=5)
    if found:
        print('\nAuto-detected project root:', found)
        PROJECT_ROOT = found
    else:
        raise FileNotFoundError(
            'Set PROJECT_ROOT manually to the folder that contains `src/` and `pyADM1ODE/`.\n'
            'In Colab: Files icon (left) → browse My Drive → open your project folder →\n'
            'click the three dots next to the folder name → "Copy path" → paste above as PROJECT_ROOT.'
        )

if not _is_project_root(PROJECT_ROOT):
    raise FileNotFoundError(
        f'Not a valid BioOptima root (need src/ and pyADM1ODE/): {PROJECT_ROOT}'
    )

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('OK — CWD:', os.getcwd())

In [ ]:
# 2. Dependencies (tqdm + plots + widgets for the live dashboard)
%pip install -q numpy pandas scipy PyYAML pyDOE2 tqdm matplotlib ipywidgets

In [ ]:
# 3. Verify PyADM1ODE is importable
pyadm_path = os.path.join(PROJECT_ROOT, 'pyADM1ODE')
if os.path.isdir(pyadm_path) and pyadm_path not in sys.path:
    sys.path.insert(0, pyadm_path)

try:
    from pyadm1 import BiogasPlant  # noqa: F401
    print('PyADM1ODE imported successfully.')
except ImportError as e:
    raise RuntimeError(
        f'Cannot import PyADM1ODE. Ensure pyADM1ODE/ is inside {PROJECT_ROOT}. Error: {e}'
    )

In [ ]:
# 4. Smoke test (~30s) — proves one full ODE run works before the long sweep
from src.sweep import _sample_params, _run_one

test_params = _sample_params(1, seed=99)[0]
test_row = _run_one(test_params, sim_days=30, V_liq=100.0)
print('Smoke test result:')
print(f"  converged={test_row['converged']}, q_ch4_avg={test_row['q_ch4_avg']:.3f}, pH={test_row['pH_final']:.2f}")
assert test_row['converged'], 'Smoke test did not converge — check substrate YAMLs and PyADM1ODE install.'
print('OK — safe to start the full sweep.')

## 5. Full sweep — live dashboard + tqdm

Adjust `N_SAMPLES`, `WORKERS`, and `AVG_SEC_PER_SIM` (your rough guess per simulation) to tune ETA text. The tqdm bar gets real ETA from actual wall time once enough jobs finish.

In [ ]:
# --- config ---
import time
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML, Audio

from src.sweep import generate_dataset

N_SAMPLES = 3000
WORKERS = 2  # Colab free tier: 2 CPUs typical
V_LIQ = 100.0
SIM_DAYS = 30
SEED = 42

# Rough guess for *first* ETA line only (tqdm replaces with real ETA quickly)
AVG_SEC_PER_SIM = 30.0

OUT_CSV = Path(PROJECT_ROOT) / 'data' / 'generated' / 'biogas_training_data.csv'

# How often to redraw charts / HTML (every N finished sims)
REFRESH_EVERY = 15
# Console heartbeat
PRINT_EVERY = 25

%matplotlib inline

q_hist = []
conv_hist = []
last_errors = deque(maxlen=6)
milestones_played = set()


def _tone(freq_hz: float = 880.0, duration_s: float = 0.18, vol: float = 0.25):
    sr = 8000
    t = np.linspace(0.0, duration_s, int(sr * duration_s), endpoint=False)
    wave = (vol * np.sin(2.0 * np.pi * freq_hz * t)).astype(np.float32)
    return Audio(wave, rate=sr, autoplay=True)


fig, (ax_c, ax_q) = plt.subplots(1, 2, figsize=(11, 3.2))
fig.suptitle('Live sweep monitor', fontsize=12)
plot_handle = display(fig, display_id=True)

status_handle = display(
    HTML("<div style='font-family:system-ui;padding:8px;border:1px solid #ccc;border-radius:8px;background:#f8fafc'>Starting…</div>"),
    display_id=True,
)


def _play_milestone(done: int, total: int) -> None:
    frac = done / max(total, 1)
    for m, label in [(0.25, 25), (0.50, 50), (0.75, 75), (1.00, 100)]:
        if frac >= m and label not in milestones_played:
            milestones_played.add(label)
            try:
                display(_tone(660.0 if label != 100 else 520.0, 0.22, 0.22))
            except Exception:
                pass


def _redraw(done: int, total: int, elapsed_s: float) -> None:
    ax_c.clear()
    ax_q.clear()
    xs = np.arange(1, len(conv_hist) + 1)
    if len(conv_hist):
        csum = np.cumsum(conv_hist)
        rate = csum / xs
        ax_c.plot(xs, rate, color='#2563eb', lw=1.8)
        ax_c.set_ylim(0, 1.05)
        ax_c.set_xlabel('Finished simulations')
        ax_c.set_ylabel('Cumulative convergence rate')
        ax_c.grid(True, alpha=0.3)
    if len(q_hist):
        tail = min(200, len(q_hist))
        ax_q.plot(np.arange(len(q_hist) - tail, len(q_hist)), q_hist[-tail:], color='#059669', lw=1.2)
        ax_q.set_xlabel('Last N finished jobs')
        ax_q.set_ylabel('q_ch4_avg (last run)')
        ax_q.grid(True, alpha=0.3)
    fig.tight_layout()
    plot_handle.update(fig)

    conv_rate = float(np.mean(conv_hist)) if conv_hist else 0.0
    eta_s = (elapsed_s / max(done, 1)) * (total - done)
    err_html = ""
    if last_errors:
        err_html = "<br/><b>Recent sim_error snippets:</b><br/>" + "<br/>".join(last_errors)
    html = f"""
    <div style="font-family:system-ui;padding:10px 12px;border:1px solid #cbd5e1;border-radius:10px;background:linear-gradient(180deg,#f8fafc,#eef2ff)">
      <div style="display:flex;flex-wrap:wrap;gap:14px;align-items:center">
        <div><b>Progress</b><br/><span style="font-size:22px">{done} / {total}</span></div>
        <div><b>Elapsed</b><br/>{elapsed_s/3600:.2f} h</div>
        <div><b>ETA (rolling)</b><br/>{eta_s/3600:.2f} h</div>
        <div><b>Conv. so far</b><br/>{conv_rate*100:.1f}%</div>
        <div><b>Last q_ch4</b><br/>{q_hist[-1]:.2f}</div>
        <div><b>Last pH / VFA</b><br/>{last_row.get('pH_final', float('nan')):.2f} / {last_row.get('VFA_final', float('nan')):.2f}</div>
      </div>
      {err_html}
    </div>
    """
    status_handle.update(HTML(html))


last_row = {}


def progress_hook(info: dict) -> None:
    global last_row
    done = int(info['done'])
    total = int(info['total'])
    row = info['row']
    last_row = row
    elapsed_s = float(info['elapsed_s'])

    q_hist.append(float(row.get('q_ch4_avg', 0.0)))
    conv_hist.append(1.0 if bool(row.get('converged')) else 0.0)
    err = row.get('sim_error')
    if err:
        last_errors.appendleft(str(err)[:160])

    if done % PRINT_EVERY == 0 or done == total:
        print(
            f"[{done:4d}/{total}] elapsed={elapsed_s/60:6.1f} min | "
            f"conv={bool(row.get('converged'))} | q_ch4={float(row.get('q_ch4_avg', 0.0)):8.2f} | idx={row.get('idx')}"
        )

    if done % REFRESH_EVERY == 0 or done == total:
        _redraw(done, total, elapsed_s)

    _play_milestone(done, total)


guess_hours = N_SAMPLES * AVG_SEC_PER_SIM / max(WORKERS, 1) / 3600.0
print(f"Starting sweep: N={N_SAMPLES}, workers={WORKERS}, out={OUT_CSV}")
print(f"Ballpark duration (rough): ~{guess_hours:.1f} h — tqdm + dashboard will refine this.")

t0 = time.perf_counter()
generate_dataset(
    N_SAMPLES,
    OUT_CSV,
    sim_days=SIM_DAYS,
    V_liq=V_LIQ,
    seed=SEED,
    workers=WORKERS,
    progress_bar=True,
    progress_hook=progress_hook,
)
wall = time.perf_counter() - t0
print(f"Done in {wall/3600:.2f} h. Wrote: {OUT_CSV}")
display(HTML(f"<h3 style='color:#15803d'>Finished — {N_SAMPLES} rows saved.</h3>"))
try:
    display(_tone(523.0, 0.35, 0.28))
except Exception:
    pass

In [ ]:
# 6. Sanity check (read back the CSV)
df = pd.read_csv(OUT_CSV)
n_converged = int(df['converged'].sum())
print(f"Rows: {len(df)} | converged: {n_converged} ({n_converged/len(df)*100:.0f}%)")
print(df[['q_ch4_avg', 'pH_final', 'VFA_final', 'stability_label']].describe().round(3))

olr_col = df['OLR'] if 'OLR' in df.columns else df.get('OLR_used', pd.Series(dtype=float))
print(f"\nOLR range: {olr_col.min():.2f} – {olr_col.max():.2f}")
in_ui = ((olr_col >= 0.5) & (olr_col <= 6.0)).mean() * 100
print(f"Rows in UI OLR band [0.5, 6.0]: {in_ui:.1f}%")
print(f"HRT range: {df['HRT_days'].min():.1f} – {df['HRT_days'].max():.1f}")
label_map = {0: 'Stable', 1: 'Warning', 2: 'Critical'}
valid = df[df['stability_label'] >= 0]
print(f"Stability: {dict(valid['stability_label'].map(label_map).value_counts())}")

fig2, axes = plt.subplots(1, 3, figsize=(14, 3.5))
df['q_ch4_avg'].hist(bins=40, ax=axes[0], color='#6366f1', edgecolor='white')
axes[0].set_title('CH₄ yield'); axes[0].set_xlabel('q_ch4_avg')
olr_col.hist(bins=40, ax=axes[1], color='#2d6a4f', edgecolor='white')
axes[1].axvline(0.5, color='red', ls='--'); axes[1].axvline(6.0, color='red', ls='--')
axes[1].set_title('OLR (red = UI bounds)'); axes[1].set_xlabel('OLR')
valid['stability_label'].map(label_map).value_counts().reindex(['Stable','Warning','Critical']).plot.bar(
    ax=axes[2], color=['#2d6a4f','#e9c46a','#e76f51'])
axes[2].set_title('Stability classes')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Download CSV
from google.colab import files

files.download(str(OUT_CSV))
print('Download started. Copy into your repo as data/generated/biogas_training_data.csv')

## Supplementary sweeps (low OLR + instability)

Run the cells below **after** the main 3000-run sweep is done and downloaded.

**What these do:**
1. **Low OLR sweep** (1200 runs, ~40–90 min) — fills the OLR 0.5–2.5 gap using diluted substrates
2. **Instability sweep** (500 runs, ~20–40 min) — generates Warning/Critical labels via extreme conditions

**Output files** (saved inside your project on Google Drive):
- `data/generated/biogas_training_low_olr.csv`
- `data/generated/biogas_training_instability.csv`

After both finish, download them and copy to `D:\major project\webapp\data\generated\` on your PC.

In [ ]:
# 8a. Create low-OLR substrate YAMLs (TS halved → lower OLR at same HRT)
import time, os

print("=" * 60)
print("STEP 1/4: Creating low-OLR substrate YAMLs...")
print("=" * 60)
!python scripts/create_low_olr_substrates.py

# Verify they exist
low_olr_dir = os.path.join(PROJECT_ROOT, "data", "substrates", "low_olr")
yamls = [f for f in os.listdir(low_olr_dir) if f.endswith(".yaml")] if os.path.isdir(low_olr_dir) else []
print(f"\n✅ Created {len(yamls)} low-OLR YAML files in: {low_olr_dir}")
for y in sorted(yamls):
    print(f"   {y}")

In [ ]:
# 8b. Low-OLR sweep (1200 samples, ~40–90 min)
#     Targets OLR 0.5–2.5 using diluted substrates + longer HRT
import time, os, pandas as pd

LOW_OLR_CSV = os.path.join(PROJECT_ROOT, "data", "generated", "biogas_training_low_olr.csv")

print("=" * 60)
print("STEP 2/4: Running LOW-OLR sweep (1200 samples)...")
print("  Target OLR: 0.5 – 2.5 kg VS/m³/d")
print("  HRT range:  25 – 60 days")
print("  Substrates: data/substrates/low_olr/*.yaml (TS × 0.55)")
print(f"  Output:     {LOW_OLR_CSV}")
print("=" * 60)

t0 = time.time()
!python scripts/generate_supplementary_sweeps.py --mode low_olr --progress
elapsed = time.time() - t0

print(f"\n⏱️  Low-OLR sweep finished in {elapsed/60:.1f} minutes")

if os.path.isfile(LOW_OLR_CSV):
    df = pd.read_csv(LOW_OLR_CSV)
    conv = df[df["converged"] == True] if "converged" in df.columns else df
    print(f"✅ File saved: {LOW_OLR_CSV}")
    print(f"   Rows: {len(df)} | Converged: {len(conv)}")
    print(f"   OLR range: {conv['OLR'].min():.2f} – {conv['OLR'].max():.2f}")
    stab = conv["stability_label"].value_counts().to_dict() if "stability_label" in conv.columns else {}
    print(f"   Stability: {stab}")
else:
    print("❌ ERROR: Low-OLR CSV was NOT created. Check errors above.")

In [ ]:
# 8c. Instability sweep (500 samples, ~20–40 min)
#     Short HRT + food-heavy mixes → high OLR, stressed digester
import time, os, pandas as pd

INSTAB_CSV = os.path.join(PROJECT_ROOT, "data", "generated", "biogas_training_instability.csv")

print("=" * 60)
print("STEP 3/4: Running INSTABILITY sweep (500 samples)...")
print("  HRT range:  10 – 18 days (short, stressed)")
print("  Mix:        Food-waste-heavy Dirichlet")
print("  Goal:       Generate Warning/Critical stability labels")
print(f"  Output:     {INSTAB_CSV}")
print("=" * 60)

t0 = time.time()
!python scripts/generate_supplementary_sweeps.py --mode instability --progress
elapsed = time.time() - t0

print(f"\n⏱️  Instability sweep finished in {elapsed/60:.1f} minutes")

if os.path.isfile(INSTAB_CSV):
    df = pd.read_csv(INSTAB_CSV)
    conv = df[df["converged"] == True] if "converged" in df.columns else df
    label_map = {0: "Stable", 1: "Warning", 2: "Critical", -1: "Failed"}
    print(f"✅ File saved: {INSTAB_CSV}")
    print(f"   Rows: {len(df)} | Converged: {len(conv)}")
    print(f"   OLR range: {conv['OLR'].min():.2f} – {conv['OLR'].max():.2f}")
    stab = conv["stability_label"].map(label_map).value_counts().to_dict()
    print(f"   Stability: {stab}")
    if 1 not in conv["stability_label"].values and 2 not in conv["stability_label"].values:
        print("   ⚠️  No Warning/Critical labels generated — may need more extreme conditions")
    else:
        print("   🎯 Non-stable samples found!")
else:
    print("❌ ERROR: Instability CSV was NOT created. Check errors above.")

In [ ]:
# 8d. Download LOW-OLR CSV
import os
from google.colab import files

LOW_OLR_CSV = os.path.join(PROJECT_ROOT, "data", "generated", "biogas_training_low_olr.csv")

if os.path.isfile(LOW_OLR_CSV):
    print(f"📥 Downloading: {os.path.basename(LOW_OLR_CSV)}")
    files.download(LOW_OLR_CSV)
    print("✅ Done — save to:  D:\\major project\\webapp\\data\\generated\\biogas_training_low_olr.csv")
else:
    print(f"❌ File not found: {LOW_OLR_CSV}")
    print("   Re-run cell 8b first.")

In [ ]:
# 8e. Download INSTABILITY CSV
import os
from google.colab import files

INSTAB_CSV = os.path.join(PROJECT_ROOT, "data", "generated", "biogas_training_instability.csv")

if os.path.isfile(INSTAB_CSV):
    print(f"📥 Downloading: {os.path.basename(INSTAB_CSV)}")
    files.download(INSTAB_CSV)
    print("✅ Done — save to:  D:\\major project\\webapp\\data\\generated\\biogas_training_instability.csv")
else:
    print(f"❌ File not found: {INSTAB_CSV}")
    print("   Re-run cell 8c first.")

print()
print("=" * 60)
print("Both CSVs downloaded? On your PC run:")
print("  python scripts/merge_training_csvs.py")
print("  python scripts/train_models.py --csv data/generated/biogas_training_merged.csv")
print("  python scripts/validate_against_literature.py")
print("=" * 60)